In [1]:
# =========================================
# Colab starter: load data + run regressions
# =========================================

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col

# -------------------------
# 1) Load uploaded CSV
# -------------------------
# Upload Baseline.csv via the file browser, then:
path = "/content/Baseline.csv"   # change if needed
df = pd.read_csv(path)

print("Shape:", df.shape)
print("Columns (sample):", list(df.columns)[:25])

# -------------------------
# 2) Basic cleaning / types
# -------------------------
# Parse datetime (your file has timezone in the string; this handles it)
df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce", utc=True)

# If you want ET fixed effects, convert to US/Eastern (optional)
# df["datetime_et"] = df["datetime"].dt.tz_convert("US/Eastern")
# Use hour/month/year columns that already exist in dataset.

# Drop rows missing key stuff (adjust as you want)
key_cols = ["symbol", "user_name", "ret_5m", "ret_30m", "vol_sym_5m", "vol_sym_30m",
            "overall_sentiment", "favorites", "retweets", "hour", "month", "year"]
df = df.dropna(subset=[c for c in key_cols if c in df.columns]).copy()

# Keep only the 3 figures
keep_figs = ["Donald Trump", "Elon Musk", "Joe Biden"]
df = df[df["user_name"].isin(keep_figs)].copy()

# -------------------------
# 3) Construct regressors
# -------------------------
# Engagement logs
df["log_retweets"]   = np.log1p(df["retweets"])
df["log_favorites"]  = np.log1p(df["favorites"])

# Keyword intensity: sum all *_hits columns
# Keyword intensity: sum all *_hits columns
hit_cols = [c for c in df.columns if c.endswith("_hits")]

if hit_cols:
    # Force numeric (convert strings like "1" to 1)
    df[hit_cols] = df[hit_cols].apply(pd.to_numeric, errors="coerce")
    df["keyword_intensity"] = df[hit_cols].sum(axis=1)
else:
    df["keyword_intensity"] = 0.0
# Optional: include pre-controls if you want them in X_i
# (you have pre_ret_5m, pre_ret_30m, vol_pre_5m, vol_pre_30m)
# define them by horizon.
def horizon_cols(h):
    if h == "5m":
        return {"ret": "ret_5m", "vol": "vol_sym_5m",
                "pre_ret": "pre_ret_5m" if "pre_ret_5m" in df.columns else None,
                "pre_vol": "vol_pre_5m" if "vol_pre_5m" in df.columns else None}
    if h == "30m":
        return {"ret": "ret_30m", "vol": "vol_sym_30m",
                "pre_ret": "pre_ret_30m" if "pre_ret_30m" in df.columns else None,
                "pre_vol": "vol_pre_30m" if "vol_pre_30m" in df.columns else None}
    raise ValueError("h must be '5m' or '30m'")

# Time fixed effects
# use year-month and hour FE; you can swap in date FE if you prefer.
df["ym"] = df["year"].astype(int).astype(str) + "-" + df["month"].astype(int).astype(str).str.zfill(2)
df["hour_fe"] = df["hour"].astype(int)

# -------------------------
# 4) Regression runners
# -------------------------
def run_ols(formula, data, cluster_col=None):
    """
    Runs OLS with robust or clustered SE.
    - If cluster_col is provided, clusters SEs by that column.
    """
    model = smf.ols(formula, data=data)
    if cluster_col is not None and cluster_col in data.columns:
        res = model.fit(cov_type="cluster", cov_kwds={"groups": data[cluster_col]})
    else:
        res = model.fit(cov_type="HC1")  # robust (heteroskedasticity-consistent)
    return res

def build_controls(h):
    """
    X_i controls used across specs:
    - engagement logs
    - keyword_intensity (optional; you can remove if you want it only in the keyword regression)
    - pre-return and pre-vol (if available)
    - time fixed effects (hour + year-month)
    """
    cols = horizon_cols(h)

    controls = ["log_retweets", "log_favorites", "C(hour_fe)", "C(ym)"]

    # add pre-controls if present
    if cols["pre_ret"] is not None:
        controls.append(cols["pre_ret"])
    if cols["pre_vol"] is not None:
        controls.append(cols["pre_vol"])

    return " + ".join(controls)

def run_all_specs_for_symbol(symbol, h="5m", y_kind="ret", sentiment_col="overall_sentiment"):
    """
    Runs the four specs matching your Analysis:
    4.3 Baseline identity effect
    4.4 Sentiment effects with interactions
    Engagement-only spec
    Keyword intensity spec
    """
    cols = horizon_cols(h)
    y = cols["ret"] if y_kind == "ret" else cols["vol"]
    d = df[df["symbol"] == symbol].copy()

    X = build_controls(h)

    # NOTE: Using C(user_name) is standard way to do 1[Trump], 1[Musk], 1[Biden] indicators
    # with one omitted as the reference to avoid perfect multicollinearity.

    # 4.3 Baseline Identity Spec (figure indicators + controls)
    f_base = f"{y} ~ C(user_name) + {X}"

    # 4.4 Sentiment + interactions (sentiment, figure dummies, sentiment*figure + controls)
    f_sent = f"{y} ~ {sentiment_col} + C(user_name) + {sentiment_col}:C(user_name) + {X}"

    # Engagement spec (as you wrote)
    f_eng  = f"{y} ~ log_retweets + log_favorites + {X}"

    # Keyword intensity spec (keyword intensity + figure dummies + controls)
    f_kw   = f"{y} ~ keyword_intensity + C(user_name) + {X}"

    # Choose clustering (optional):
    # Common choice: cluster by day (needs a day variable). If you have 'date', use that:
    cluster_col = "date" if "date" in d.columns else None

    r_base = run_ols(f_base, d, cluster_col=cluster_col)
    r_sent = run_ols(f_sent, d, cluster_col=cluster_col)
    r_eng  = run_ols(f_eng,  d, cluster_col=cluster_col)
    r_kw   = run_ols(f_kw,   d, cluster_col=cluster_col)

    return (r_base, r_sent, r_eng, r_kw)

# -------------------------
# 5) Run (SPY/QQQ) x (5m/30m) x (return/vol)
# -------------------------
symbols = sorted(df["symbol"].unique().tolist())
print("Symbols found:", symbols)

def print_models(models, title):
    info = {"N": lambda x: f"{int(x.nobs)}",
            "R2": lambda x: f"{x.rsquared:.3f}"}
    table = summary_col(list(models),
                        model_names=["Baseline", "Sent+Int", "Engage", "Keyword"],
                        stars=True,
                        info_dict=info)
    print("\n" + "="*80)
    print(title)
    print(table)

for sym in symbols:
    for h in ["5m", "30m"]:
        # Returns
        m = run_all_specs_for_symbol(sym, h=h, y_kind="ret", sentiment_col="overall_sentiment")
        print_models(m, title=f"{sym} | h={h} | DepVar=Return")

        # Volatility
        m = run_all_specs_for_symbol(sym, h=h, y_kind="vol", sentiment_col="overall_sentiment")
        print_models(m, title=f"{sym} | h={h} | DepVar=Volatility")

Shape: (718, 74)
Columns (sample): ['id', 'text', 'favorites', 'retweets', 'date', 'original_tweet', 'datetime', 'words', 'errors', 'errors_count', 'words_count', 'sentence_length', 'hour', 'month', 'year', 'Vader sentiment', 'user_name', 'finbert_sentiment', 'avg_sentiment', 'overall_sentiment', 'kw_geopolitical_trade_hits', 'kw_geopolitical_trade_flag', 'kw_monetary_fed_inflation_hits', 'kw_monetary_fed_inflation_flag', 'kw_fiscal_regulatory_hits']
Symbols found: ['QQQ', 'SPY']

QQQ | h=5m | DepVar=Return

                                                         Baseline  Sent+Int   Engage   Keyword 
-----------------------------------------------------------------------------------------------
Intercept                                               -0.0009   -0.0008   -0.0002   -0.0009  
                                                        (0.0007)  (0.0008)  (0.0003)  (0.0007) 
C(user_name)[T.Elon Musk]                               0.0005    0.0006              0.0005   
      

In [6]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
df["year"] = pd.to_datetime(df["datetime"]).dt.year
# -----------------------
# Helper: run OLS w/ SEs
# -----------------------
def run_ols(formula, data, cluster_col=None):
    model = smf.ols(formula, data=data)
    if cluster_col is not None and cluster_col in data.columns:
        return model.fit(cov_type="cluster", cov_kwds={"groups": data[cluster_col]})
    return model.fit(cov_type="HC1")  # robust SE

# -----------------------
# Helper: build formulas
# -----------------------
def horizon_cols(df, h):
    if h == "5m":
        return {"ret": "ret_5m", "vol": "vol_sym_5m",
                "pre_ret": "pre_ret_5m" if "pre_ret_5m" in df.columns else None,
                "pre_vol": "vol_pre_5m" if "vol_pre_5m" in df.columns else None}
    if h == "30m":
        return {"ret": "ret_30m", "vol": "vol_sym_30m",
                "pre_ret": "pre_ret_30m" if "pre_ret_30m" in df.columns else None,
                "pre_vol": "vol_pre_30m" if "vol_pre_30m" in df.columns else None}
    raise ValueError("h must be '5m' or '30m'")

def build_controls(df, h):
    cols = horizon_cols(df, h)
    controls = ["log_retweets", "log_favorites", "C(hour_fe)", "C(year)"]  # <-- CHANGED HERE
    if cols["pre_ret"] is not None: controls.append(cols["pre_ret"])
    if cols["pre_vol"] is not None: controls.append(cols["pre_vol"])
    return " + ".join(controls)
# ---------------------------------------
# Helper: pretty regression table (small)
# ---------------------------------------
def stars(p):
    if p < 0.01: return "***"
    if p < 0.05: return "**"
    if p < 0.10: return "*"
    return ""

def tidy_res(res, keep_substrings, rename_map=None):
    out = pd.DataFrame({
        "term": res.params.index.astype(str),
        "coef": res.params.values,
        "se": res.bse.values,
        "p": res.pvalues.values
    })

    mask = np.zeros(len(out), dtype=bool)
    for k in keep_substrings:
        mask |= out["term"].str.contains(k, regex=False)
    out = out[mask].copy()

    # ONE-LINE formatting (no newlines)
    out["est"] = out.apply(
        lambda r: f'{r["coef"]:.4f}{stars(r["p"])} ({r["se"]:.4f})', axis=1
    )

    if rename_map:
        out["term"] = out["term"].replace(rename_map)

    return out[["term", "est"]].set_index("term")

def make_clean_table(results_dict, keep_substrings, rename_map=None):
    tabs = []
    for name, res in results_dict.items():
        t = tidy_res(res, keep_substrings, rename_map).rename(columns={"est": name})
        tabs.append(t)
    tab = pd.concat(tabs, axis=1)
    return tab

# ---------------------------------------------------
# ONE FUNCTION: run models + print readable output
# ---------------------------------------------------
def run_and_display(df, symbol, h, depvar, sentiment_col="overall_sentiment"):
    """
    depvar: 'ret' or 'vol'
    """

    d = df[df["symbol"] == symbol].copy()
    cols = horizon_cols(d, h)
    y = cols["ret"] if depvar == "ret" else cols["vol"]

    X = build_controls(d, h)

    # Model formulas
    f_base = f"{y} ~ C(user_name) + {X}"
    f_sent = f"{y} ~ {sentiment_col} + C(user_name) + {sentiment_col}:C(user_name) + {X}"
    f_eng  = f"{y} ~ log_retweets + log_favorites + {X}"
    f_kw   = f"{y} ~ keyword_intensity + C(user_name) + {X}"

    cluster_col = "date" if "date" in d.columns else None

    r_base = run_ols(f_base, d, cluster_col=cluster_col)
    r_sent = run_ols(f_sent, d, cluster_col=cluster_col)
    r_eng  = run_ols(f_eng,  d, cluster_col=cluster_col)
    r_kw   = run_ols(f_kw,   d, cluster_col=cluster_col)

    results = {"Baseline": r_base, "Sent+Int": r_sent, "Engage": r_eng, "Keyword": r_kw}

    KEEP = [
        "Intercept",
        "C(user_name)",
        sentiment_col,
        f"{sentiment_col}:C(user_name)",
        "log_retweets", "log_favorites",
        "keyword_intensity",
        "pre_ret_5m", "pre_ret_30m",
        "vol_pre_5m", "vol_pre_30m"
    ]

    RENAME = {
        "Intercept": "Constant",
        "log_retweets": "log(1+Retweets)",
        "log_favorites": "log(1+Favorites)",
        "keyword_intensity": "Keyword intensity",
        "pre_ret_5m": "Pre-return (5m)",
        "pre_ret_30m": "Pre-return (30m)",
        "vol_pre_5m": "Pre-vol (5m)",
        "vol_pre_30m": "Pre-vol (30m)",
        "C(user_name)[T.Elon Musk]": "Elon Musk (vs Trump)",
        "C(user_name)[T.Joe Biden]": "Joe Biden (vs Trump)",
        f"{sentiment_col}[T.Neutral]": "Neutral (vs Negative)",
        f"{sentiment_col}[T.Positive]": "Positive (vs Negative)",
        f"{sentiment_col}[T.Neutral]:C(user_name)[T.Elon Musk]": "Neutral × Musk",
        f"{sentiment_col}[T.Positive]:C(user_name)[T.Elon Musk]": "Positive × Musk",
        f"{sentiment_col}[T.Neutral]:C(user_name)[T.Joe Biden]": "Neutral × Biden",
        f"{sentiment_col}[T.Positive]:C(user_name)[T.Joe Biden]": "Positive × Biden",
    }

    tab = make_clean_table(results, KEEP, RENAME).fillna("")

    # Add N and R2 rows
    info = pd.DataFrame({
    "Baseline": [int(r_base.nobs), round(r_base.rsquared,4), "Yes"],
    "Sent+Int": [int(r_sent.nobs), round(r_sent.rsquared,4), "Yes"],
    "Engage":   [int(r_eng.nobs),  round(r_eng.rsquared,4), "Yes"],
    "Keyword":  [int(r_kw.nobs),   round(r_kw.rsquared,4), "Yes"],
}, index=["Observations", "$R^2$", "Year FE"])

    tab_full = pd.concat([tab, info])

    print(f"\n=== {symbol} | h={h} | DepVar={depvar.upper()} ===")
    display(tab_full)

    # ---------- LATEX EXPORT ----------
    caption = f"{symbol}: {h}-minute {'return' if depvar=='ret' else 'volatility'} responses"
    label = f"tab:{symbol.lower()}_{depvar}_{h}"

    latex_body = tab_full.to_latex(
        escape=False,
        column_format="lcccc"
    )

    latex_table = f"""
\\begin{{table}}[!htbp]
\\centering
\\caption{{{caption}}}
\\label{{{label}}}
\\begin{{adjustbox}}{{max width=\\textwidth}}
{latex_body}
\\end{{adjustbox}}
\\begin
\\footnotesize
Notes: Heteroskedasticity-robust standard errors in parentheses.
All specifications include year fixed effects; coefficients are omitted for brevity.
\\end
"""

    print("\n\n---------- COPY LATEX BELOW ----------\n")
    print(latex_table)

    return results

# ---------------------------
# ---------------------------
# Returns
_ = run_and_display(df, symbol="SPY", h="30m", depvar="ret", sentiment_col="overall_sentiment")
# Volatility
_ = run_and_display(df, symbol="SPY", h="30m", depvar="vol", sentiment_col="overall_sentiment")


=== SPY | h=30m | DepVar=RET ===


,Baseline,Sent+Int,Engage,Keyword
Constant,-0.0001 (0.0013),-0.0006 (0.0010),0.0002 (0.0007),0.0018 (0.0023)
Elon Musk (vs Trump),0.0001 (0.0007),0.0009 (0.0009),,-0.0006 (0.0009)
Joe Biden (vs Trump),0.0002 (0.0008),0.0001 (0.0007),,-0.0003 (0.0010)
log(1+Retweets),0.0001 (0.0002),0.0001 (0.0002),0.0001 (0.0002),0.0001 (0.0002)
log(1+Favorites),-0.0001 (0.0001),-0.0001 (0.0001),-0.0001 (0.0001),-0.0001 (0.0000)
Pre-return (30m),-0.0203 (0.2892),-0.0649 (0.2694),-0.0167 (0.2812),-0.0281 (0.2854)
Pre-vol (30m),7.3172 (29.1730),14.4680 (36.4024),7.5349 (28.7238),5.8906 (28.7117)
Neutral (vs Negative),,-0.0002 (0.0004),,
Positive (vs Negative),,-0.0004 (0.0003),,
Neutral × Musk,,-0.0009 (0.0011),,




---------- COPY LATEX BELOW ----------


\begin{table}[!htbp]
\centering
\caption{SPY: 30m-minute return responses}
\label{tab:spy_ret_30m}
\begin{adjustbox}{max width=\textwidth}
\begin{tabular}{lcccc}
\toprule
 & Baseline & Sent+Int & Engage & Keyword \\
\midrule
Constant & -0.0001 (0.0013) & -0.0006 (0.0010) & 0.0002 (0.0007) & 0.0018 (0.0023) \\
Elon Musk (vs Trump) & 0.0001 (0.0007) & 0.0009 (0.0009) &  & -0.0006 (0.0009) \\
Joe Biden (vs Trump) & 0.0002 (0.0008) & 0.0001 (0.0007) &  & -0.0003 (0.0010) \\
log(1+Retweets) & 0.0001 (0.0002) & 0.0001 (0.0002) & 0.0001 (0.0002) & 0.0001 (0.0002) \\
log(1+Favorites) & -0.0001 (0.0001) & -0.0001 (0.0001) & -0.0001 (0.0001) & -0.0001 (0.0000) \\
Pre-return (30m) & -0.0203 (0.2892) & -0.0649 (0.2694) & -0.0167 (0.2812) & -0.0281 (0.2854) \\
Pre-vol (30m) & 7.3172 (29.1730) & 14.4680 (36.4024) & 7.5349 (28.7238) & 5.8906 (28.7117) \\
Neutral (vs Negative) &  & -0.0002 (0.0004) &  &  \\
Positive (vs Negative) &  & -0.0004 (0.0003) &  &  \

,Baseline,Sent+Int,Engage,Keyword
Constant,-0.0000 (0.0001),0.0000 (0.0000),-0.0000 (0.0000),-0.0001 (0.0001)
Elon Musk (vs Trump),0.0000 (0.0000),-0.0000 (0.0000),,0.0000 (0.0000)
Joe Biden (vs Trump),0.0000 (0.0000),-0.0000 (0.0000),,0.0000 (0.0000)
log(1+Retweets),-0.0000 (0.0000),-0.0000 (0.0000),-0.0000 (0.0000),0.0000 (0.0000)
log(1+Favorites),0.0000 (0.0000),0.0000 (0.0000),0.0000 (0.0000),0.0000 (0.0000)
Pre-return (30m),-0.0090 (0.0103),-0.0068 (0.0078),-0.0086 (0.0099),-0.0086 (0.0100)
Pre-vol (30m),0.1033 (0.8642),-0.5638 (1.5634),0.1269 (0.8355),0.1654 (0.8135)
Neutral (vs Negative),,0.0000 (0.0000),,
Positive (vs Negative),,-0.0000 (0.0000),,
Neutral × Musk,,-0.0000 (0.0000),,




---------- COPY LATEX BELOW ----------


\begin{table}[!htbp]
\centering
\caption{SPY: 30m-minute volatility responses}
\label{tab:spy_vol_30m}
\begin{adjustbox}{max width=\textwidth}
\begin{tabular}{lcccc}
\toprule
 & Baseline & Sent+Int & Engage & Keyword \\
\midrule
Constant & -0.0000 (0.0001) & 0.0000 (0.0000) & -0.0000 (0.0000) & -0.0001 (0.0001) \\
Elon Musk (vs Trump) & 0.0000 (0.0000) & -0.0000 (0.0000) &  & 0.0000 (0.0000) \\
Joe Biden (vs Trump) & 0.0000 (0.0000) & -0.0000 (0.0000) &  & 0.0000 (0.0000) \\
log(1+Retweets) & -0.0000 (0.0000) & -0.0000 (0.0000) & -0.0000 (0.0000) & 0.0000 (0.0000) \\
log(1+Favorites) & 0.0000 (0.0000) & 0.0000 (0.0000) & 0.0000 (0.0000) & 0.0000 (0.0000) \\
Pre-return (30m) & -0.0090 (0.0103) & -0.0068 (0.0078) & -0.0086 (0.0099) & -0.0086 (0.0100) \\
Pre-vol (30m) & 0.1033 (0.8642) & -0.5638 (1.5634) & 0.1269 (0.8355) & 0.1654 (0.8135) \\
Neutral (vs Negative) &  & 0.0000 (0.0000) &  &  \\
Positive (vs Negative) &  & -0.0000 (0.0000) &  &  \\

In [ ]:
!pip -q install stargazer